# Boundary and Subdomain Mesh Preparation for OGS

    This notebook prepares the mesh information required by OpenGeoSys (OGS) to correctly identify the physical regions of the model. 

    The geometry and mesh were created in Gmsh (see "01_gmsh_mesh_generation.ipynb"). Each layer, well interval, and boundary was assigned a physical tag. OGS uses these tags to assign properties, boundary conditions, and source terms.

    The notebook is organized into the following main sections:

    1. Define physical volume tags
    2. Create 3D VTU Submesh for Each Physical Group
    3. Create MaterialIDs
    4. Create 2D VTU Surface Mesh per Physical Group

# 1. Define physical volume tags 

    The main objective is to read the bulk mesh generated previously, recover the physical tags assigned in Gmsh, and use those tags to assign material properties, boundary conditions, and source or sink terms consistently in OGS.

In [2]:
# ----------------------------------------------------------------------------------------
#  Bulk mesh file `mesh6.vtu`
#    The bulk mesh is the main mesh of the model. It contains all volumetric regions, 
#    including geological layers and well sections. 
#    
# ----------------------------------------------------------------------------------------
import meshio
import numpy as np
from pathlib import Path


input_dir = Path("../input")

bulk_file = input_dir / "mesh6.vtu"
bulk = meshio.read(bulk_file)

# --------------------------------------------------
# Extract tetrahedral elements and physical tags  
#    Access the tetrahedral cells of the mesh and recover the `gmsh:physical` tags associated with each element.
# --------------------------------------------------

tetra = bulk.cells_dict["tetra"]
phys_tetra = bulk.cell_data_dict["gmsh:physical"]["tetra"]

# --------------------------------------------------
# Define physical volume tags
#    Create a dictionary that links each numerical physical tag to a physical element.
# --------------------------------------------------

physical_volumes = {
    101: "Layer5_bottom",
    102: "Layer4_caprock2",
    103: "Layer3_Reservoir",
    104: "Layer2_caprock1",
    105: "Layer1_Top",

    201: "Well1_full",
    202: "Well1_upper",
    203: "Well1_middle",
    204: "Well1_bottom",

    205: "Well2_full",
    206: "Well2_upper",
    207: "Well2_middle",
    208: "Well2_bottom",
}


# 2. Create 3D VTU Submesh for Each Physical Group

    The purpose of this step is to create a `.vtu` file for each physical group. These files make it easier to keep a clear connection between the Gmsh physical tags and the OGS model structure.


In [3]:
# --------------------------------------------------
# Selecting the tetrahedral elements that belong to each group of the 'physical_volumes' dictionary.
# --------------------------------------------------
for tag, name in physical_volumes.items():

    selected = tetra[phys_tetra == tag]       # only the tetrahedral cells whose physical tag matches the current tag

    if len(selected) == 0:
        print(f"WARNING: No tetra cells found for tag {tag} - {name}")
        continue                # no empty mesh

    used_points = np.unique(selected.flatten())          # identifing all the mesh points used by the selected tetrahedral elements.
                        # selected.flatten() --> one-dimensional list of node IDs  
    
    old_to_new = {old: new for new, old in enumerate(used_points)}   # new local node IDs; renumbered starting from zero.

    new_points = bulk.points[used_points]   # extracting the coordinates of the points used by the current physical group.
    new_tetra = np.array(
        [[old_to_new[node] for node in elem] for elem in selected],   # replacing each original node ID with its new local node ID.
        dtype=np.int64
    )

    # 'bulk_node_ids' --> keeping the connection between the extracted submesh and the original bulk mesh.
    bulk_node_ids = used_points.astype(np.uint64)

# --------------------------------------------------
# New mesh object using meshio
#--------------------------------------------------
    
    submesh = meshio.Mesh( 
        points=new_points,                        # assigning the coordinates of new_points.
        cells=[("tetra", new_tetra)],             # cell type --> tetrahedral elements.
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_tetra), tag, dtype=np.int32)]           # assigning same physical tag to every tetrahedral element.
        }
    )

# --------------------------------------------------
# Defines the output file name and location
#--------------------------------------------------
    
    out_file = input_dir / f"{name}.vtu"
    meshio.write(out_file, submesh)              # new submesh to a .vtu file.

    print(f"Created: {out_file.name}")
    print("  tag:", tag)
    print("  tetra cells:", len(new_tetra))
    print("  points:", len(new_points))

Created: Layer5_bottom.vtu
  tag: 101
  tetra cells: 27144
  points: 6139
Created: Layer4_caprock2.vtu
  tag: 102
  tetra cells: 19045
  points: 5574
Created: Layer3_Reservoir.vtu
  tag: 103
  tetra cells: 61669
  points: 13456
Created: Layer2_caprock1.vtu
  tag: 104
  tetra cells: 18926
  points: 5579
Created: Layer1_Top.vtu
  tag: 105
  tetra cells: 27269
  points: 6205
Created: Well1_full.vtu
  tag: 201
  tetra cells: 194
  points: 91
Created: Well1_upper.vtu
  tag: 202
  tetra cells: 81
  points: 44
Created: Well1_middle.vtu
  tag: 203
  tetra cells: 47
  points: 28
Created: Well1_bottom.vtu
  tag: 204
  tetra cells: 66
  points: 35
Created: Well2_full.vtu
  tag: 205
  tetra cells: 194
  points: 91
Created: Well2_upper.vtu
  tag: 206
  tetra cells: 75
  points: 42
Created: Well2_middle.vtu
  tag: 207
  tetra cells: 51
  points: 29
Created: Well2_bottom.vtu
  tag: 208
  tetra cells: 68
  points: 36


# 3. Create MaterialIDs

    The original mesh exported from Gmsh contains the physical volume tags under the field `gmsh:physical`. However, OGS commonly uses a cell data field named `MaterialIDs` to assign material properties to different regions of the model.

    For this reason, the physical tags from Gmsh are copied into a new field called `MaterialIDs`. This keeps the original physical tag information while also creating the structure expected by OGS for material assignment.

    The resulting file, `mesh6_materials.vtu`, is the bulk mesh that should be referenced in the OGS project file when defining the numerical model.

In [4]:
# --------------------------------------------------
# Creating a clean bulk mesh for OGS MaterialIDs
# --------------------------------------------------

clean_mesh = meshio.Mesh(
    points=bulk.points,     # same node coordinates as the original mesh 'bulk'.
    cells=[("tetra", tetra)],     # only tetrahedral cells
    cell_data={
        "MaterialIDs": [phys_tetra.astype(np.int32)],  # assign material properties
        "gmsh:physical": [phys_tetra.astype(np.int32)],  # original physical tag info
    },
)

out_file = input_dir / "mesh6_materials.vtu"
meshio.write(out_file, clean_mesh)

print("Created clean bulk mesh:", out_file)
print("Tetra MaterialIDs:", sorted(set(phys_tetra)))

Created clean bulk mesh: F:\ADATA\LITHIUM\OGS\OGS_files\input\mesh6_materials.vtu
Tetra MaterialIDs: [np.int32(101), np.int32(102), np.int32(103), np.int32(104), np.int32(105), np.int32(201), np.int32(202), np.int32(203), np.int32(204), np.int32(205), np.int32(206), np.int32(207), np.int32(208)]


# 4. Create 2D VTU Surface Mesh per Physical Group

    This section extracts the 2D surface physical groups from the bulk mesh and saves each one as an independent `.vtu` file.

    3D tetrahedral submeshes were created for volumetric regions; this section works with triangular surface elements. These surface meshes are important because they can be used in OGS to apply boundary conditions and source or sink terms.


In [6]:
# --------------------------------------------------
# Creating 2D VTU submesh per surface physical group
# --------------------------------------------------

# List of surface 2D physical groups.
surface_volumes = {
    301: "Left",
    302: "Right",
    303: "Front",
    304: "Back",
    305: "Top",
    306: "Bottom",
    401: "Well1_top",
    402: "Well1_completion",
    403: "Well2_top",
    404: "Well2_completion"
}

triangles = bulk.cells_dict["triangle"]   # triangle --> 2D surfaces
phys_triangles = bulk.cell_data_dict["gmsh:physical"]["triangle"]  # physical tags associated with the triangular elements

for tag, name in surface_volumes.items():

    selected = triangles[phys_triangles == tag]   # creating one subset of triangles for each boundary region

    if len(selected) == 0:
        print(f"WARNING: No triangles found for tag {tag} ({name})")
        continue

    used_points = np.unique(selected.flatten())
    old_to_new = {old: new for new, old in enumerate(used_points)}
    new_points = bulk.points[used_points]    # coordinates of the selected surface points

    new_triangles = np.array(
        [[old_to_new[node] for node in tri] for tri in selected],
        dtype=np.int64
    )

    # bulk_node_ids for OGS BC mapping
    bulk_node_ids = used_points.astype(np.uint64)

# --------------------------------------------------
# New surface mesh object
# -------------------------------------------------- 
    
    submesh = meshio.Mesh(
        points=new_points,
        cells=[("triangle", new_triangles)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_triangles), tag, dtype=np.int32)]
        }
    )

    # Write VTU
    out_file = input_dir / f"{name}.vtu"
    meshio.write(out_file, submesh)

    print(f"Created {out_file.name} | triangles: {len(new_triangles)} | points: {len(new_points)}")

Created Left.vtu | triangles: 1826 | points: 967
Created Right.vtu | triangles: 1830 | points: 969
Created Front.vtu | triangles: 1830 | points: 969
Created Back.vtu | triangles: 1830 | points: 969
Created Top.vtu | triangles: 2652 | points: 1400
Created Bottom.vtu | triangles: 2636 | points: 1387
Created Well1_top.vtu | triangles: 7 | points: 8
Created Well1_completion.vtu | triangles: 50 | points: 32
Created Well2_top.vtu | triangles: 7 | points: 8
Created Well2_completion.vtu | triangles: 52 | points: 33
